In [ ]:
import pandas as pd

from Pipeline.Global.GlobalSetting import GlobalSetting

In [ ]:
models_to_evaluate = ["RELM", "RELM_CV", "RELM_CV_Ensemble"]
config_type = "Grid_Optimization" # Or whatever prefix your files actually use

abc_results_list = []

for model_name in models_to_evaluate:
    for fold_id in range(5):
        try:
            # Check your Storage/Record folder to ensure this string exactly matches your saved CSV names
            file_name = f"{config_type}_ABC_{model_name}_Fold_{fold_id}_Results"

            # Read the CSV
            df_fold = GlobalSetting.get_dataframe_from_record(file_name)

            # Inject Contextual Metadata
            df_fold["Model_Type"] = model_name
            df_fold["Fold_ID"] = fold_id

            abc_results_list.append(df_fold)
            print(f"[Success] Extracted: {model_name} - Fold {fold_id}")

        except Exception as e:
            print(f"[WARNING] Missing or corrupt data for {model_name} - Fold {fold_id}. Error: {e}")

# --- Final Data Matrix Construction ---
if abc_results_list:
    # CRITICAL FIX: Named df_abc_master_raw so Cell 3 can use it
    df_abc_master_raw = pd.concat(abc_results_list, ignore_index=True)
else:
    print("[ERROR] No result files found. Verify the Storage/Record directory path and file names.")

df_abc_master_raw

In [ ]:
# FIX: We are only grouping by Model_Type since the hyperparameter columns aren't in the CSVs
group_cols = ["Model_Type"]

# (Optional: If you need to keep the seeds separated, use: group_cols = ["Model_Type", "ABC_Seed"])

# Dynamically extract metrics, excluding the group column and the fold/seed identifiers
metric_cols = [col for col in df_abc_master_raw.columns if col not in group_cols + ["ABC_Seed", "Fold_ID", "Seed"]]

# Calculate aggregations
agg_funcs = {metric: ['mean', 'std', 'sem', 'max', 'min'] for metric in metric_cols}

df_abc_summary = df_abc_master_raw.groupby(group_cols).agg(agg_funcs).reset_index()

# Flatten Multi-Index Columns
df_abc_summary.columns = [
    f"{col[0]}_{col[1]}" if col[1] else col[0]
    for col in df_abc_summary.columns.values
]

display(df_abc_summary)

In [ ]:
# Load your standard ELM baseline results using the safe loader
baseline_file = "Model_Testing_Result"
df_baseline_summary = GlobalSetting.get_dataframe_from_record(baseline_file)

# --- THE GRAND MERGE ---
# Stack the ABC results directly underneath the Standard ELM results
final_comparison_df = pd.concat([df_baseline_summary, df_abc_summary], ignore_index=True)

# Sort the table by Model_Type so you can easily compare them
final_comparison_df = final_comparison_df.sort_values(by=["Model_Type"]).reset_index(drop=True)

# Save the ultimate FYP result table
GlobalSetting.save_dataframe_to_record(final_comparison_df, "ULTIMATE_FYP_Comparison.csv")

# Force Jupyter to display ALL columns without truncating them
pd.set_option('display.max_columns', None)

# Show the entire dataframe
final_comparison_df